# Init

In [1]:
import pandas as pd
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
MODEL_PATH = "meta-llama/Llama-3.1-8B-Instruct"  # deepseek-ai/DeepSeek-R1-Distill-Qwen-32B    meta-llama/Llama-3.1-8B-Instruct
TP_SIZE = 2  # 你的GPU数量

print(f" 正在加载 Tokenizer 和模型 {MODEL_PATH} ... ")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 初始化 vLLM 引擎
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=TP_SIZE,
    dtype="bfloat16",
    max_model_len=1024,            # 上下文截断
    gpu_memory_utilization=0.85,  # 显存预占比例
    enforce_eager=True            # 关闭 CUDA Graph 防止 OOM
)

 正在加载 Tokenizer 和模型 meta-llama/Llama-3.1-8B-Instruct ... 
INFO 05-14 16:42:13 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 1024, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
INFO 05-14 16:42:16 [model.py:549] Resolved architecture: LlamaForCausalLM
INFO 05-14 16:42:16 [model.py:1678] Using max model len 1024
INFO 05-14 16:42:16 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-14 16:42:16 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-14 16:42:16 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-14 16:42:16 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-14 16:42:29 [vllm.py:1025] Cudagra

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(Worker_TP1 pid=1373760) INFO 05-14 16:43:28 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/2)
(Worker_TP0 pid=1373759) INFO 05-14 16:44:26 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/2)
(Worker_TP1 pid=1373760) INFO 05-14 16:45:14 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/2)
(Worker_TP1 pid=1373760) INFO 05-14 16:45:14 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 151.14s
(Worker_TP0 pid=1373759) INFO 05-14 16:45:33 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/2)
(Worker_TP0 pid=1373759) INFO 05-14 16:45:33 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 169.35s
(Worker_TP0 pid=1373759) INFO 05-14 16:45:33 [default_loader.py:384] Loading weights took 169.47 seconds
(Worker_TP0 pid=1373759) INFO 05-14 16:45:34 [gpu_model_runner.py:4820] Model loading took 7.51 GiB memory and 171.270755 seconds
(Worker_TP0 pid=1373759) INFO 05-14 16:45:58 [gpu_worker.py:436] Availabl

In [3]:
test_cases = [
    {"Sub": "car", "Rel": "on", "Obj": "street", "Expected": "Yes"},

    {"Sub": "person", "Rel": "wearing", "Obj": "hat", "Expected": "Yes"},
    {"Sub": "boat", "Rel": "flying in", "Obj": "room", "Expected": "No"},
    {"Sub": "cat", "Rel": "sitting on", "Obj": "table", "Expected": "Yes"},
    {"Sub": "pizza", "Rel": "eating", "Obj": "man", "Expected": "No"},
    {"Sub": "train", "Rel": "parked on", "Obj": "track", "Expected": "Yes"},
    {"Sub": "tree", "Rel": "riding", "Obj": "bike", "Expected": "No"},
    {"Sub": "dog", "Rel": "holding", "Obj": "branch", "Expected": "Yes"},
    {"Sub": "flower", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},

    {"Sub": "logo", "Rel": "of", "Obj": "banana", "Expected": "Yes"},

    {"Sub": "person", "Rel": "along", "Obj": "towel", "Expected": "No"},
    {"Sub": "vase", "Rel": "sitting on", "Obj": "airplane", "Expected": "No"},
    {"Sub": "bag", "Rel": "across", "Obj": "street", "Expected": "No"},
    {"Sub": "leg", "Rel": "of", "Obj": "branch", "Expected": "No"},
    {"Sub": "laptop", "Rel": "of", "Obj": "screen", "Expected": "No"},
    {"Sub": "girl", "Rel": "attached to", "Obj": "book", "Expected": "No"},
    {"Sub": "man", "Rel": "eating", "Obj": "surfboard", "Expected": "No"},
    {"Sub": "man", "Rel": "says", "Obj": "snowboard", "Expected": "No"},


    # --- 陷阱 1：主客体颠倒 / 逆向逻辑 (测试模型是否只看词语相关性，不看语序) ---
    {"Sub": "plant", "Rel": "eating", "Obj": "cow", "Expected": "No"},       # 牛吃草，草不能吃牛
    {"Sub": "car", "Rel": "riding", "Obj": "boy", "Expected": "No"},         # 男孩骑车/乘车，车不能骑男孩
    {"Sub": "food", "Rel": "eating", "Obj": "girl", "Expected": "No"},       # 食物不能吃人
    {"Sub": "number", "Rel": "says", "Obj": "sign", "Expected": "No"},       # 标牌上写着数字，数字不能"写着"标牌

    # --- 陷阱 2：物理尺度与承重常识 (测试模型是否考虑物体的实际大小和重量) ---
    {"Sub": "elephant", "Rel": "sitting on", "Obj": "chair", "Expected": "No"},  # 椅子承受不了大象
    {"Sub": "bird", "Rel": "carrying", "Obj": "horse", "Expected": "No"},        # 鸟叼不动马
    {"Sub": "cup", "Rel": "holding", "Obj": "elephant", "Expected": "No"},       # 杯子装不下大象
    {"Sub": "airplane", "Rel": "parked on", "Obj": "roof", "Expected": "No"},    # 真实世界的飞机不会停在普通屋顶上
    {"Sub": "car", "Rel": "in", "Obj": "box", "Expected": "No"},                 # 排除玩具特例，真车装不进普通纸箱

    # --- 陷阱 3：过度拟人化 / 动画片隐喻 (测试模型是否把童话当成现实) ---
    {"Sub": "motorcycle", "Rel": "wearing", "Obj": "helmet", "Expected": "No"},  # 摩托车不戴头盔，骑手才戴
    {"Sub": "shoe", "Rel": "walking on", "Obj": "sidewalk", "Expected": "No"},   # 鞋子本身不会走，必须是人穿着
    {"Sub": "dog", "Rel": "using", "Obj": "laptop", "Expected": "No"},           # 狗不具备使用电脑的物理能力
    {"Sub": "cat", "Rel": "wearing", "Obj": "boot", "Expected": "No"},           # 穿靴子的猫是童话，现实中视觉不合理
    {"Sub": "flower", "Rel": "holding", "Obj": "cup", "Expected": "No"},         # 花没有手，不能拿杯子
    {"Sub": "cabinet", "Rel": "wearing", "Obj": "coat", "Expected": "No"},       # 柜子不能“穿”外套（只能是里面挂着/装有）
    {"Sub": "bench", "Rel": "watching", "Obj": "bird", "Expected": "No"},        # 长椅没有视觉，不能 watching

    # --- 陷阱 4：生物特征错乱 / 机器与生物乱接 (测试解剖学常识) ---
    {"Sub": "wing", "Rel": "part of", "Obj": "dog", "Expected": "No"},           # 狗没有翅膀
    {"Sub": "leaf", "Rel": "part of", "Obj": "bird", "Expected": "No"},          # 鸟不长叶子
    {"Sub": "leg", "Rel": "attached to", "Obj": "wheel", "Expected": "No"},      # 真实的肉腿不会长在轮子上
    {"Sub": "wheel", "Rel": "belonging to", "Obj": "horse", "Expected": "No"},   # 马没有轮子
    {"Sub": "cow", "Rel": "sitting on", "Obj": "fence", "Expected": "No"},       # 牛的骨骼结构无法像人一样“坐”在篱笆上

    # --- 陷阱 5：空间介词混淆及极端特例 (讨论过的刁钻案例) ---
    {"Sub": "train", "Rel": "parked on", "Obj": "street", "Expected": "No"},     # 火车停在铁轨上，停在街上是极端特例
    {"Sub": "boat", "Rel": "parked on", "Obj": "tree", "Expected": "No"},        # 船不会停在树上
    {"Sub": "umbrella", "Rel": "flying in", "Obj": "room", "Expected": "No"},    # 雨伞在室内飞行（魔法/风暴极端情况）
    {"Sub": "logo", "Rel": "painted on", "Obj": "horse", "Expected": "No"},      # 活体马身上直接画Logo属于极端特例

    # --- 迷惑组：看似荒谬但物理现实完全成立的 Hard Positives (测试模型是否敢说 Yes) ---
    {"Sub": "bird", "Rel": "standing on", "Obj": "cow", "Expected": "Yes"},      # 牛背鹭等鸟类停在牛背上很常见
    {"Sub": "plant", "Rel": "growing on", "Obj": "building", "Expected": "Yes"}, # 爬山虎、苔藓长在建筑上
    {"Sub": "sign", "Rel": "says", "Obj": "number", "Expected": "Yes"},          # 标牌上写着限速等数字
    {"Sub": "cat", "Rel": "looking at", "Obj": "book", "Expected": "Yes"},       # 猫虽然不认字，但物理上盯着书看完全可能
    {"Sub": "dog", "Rel": "covered in", "Obj": "snow", "Expected": "Yes"},       # 狗在雪地玩耍身上覆满雪
    {"Sub": "tree", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},      # 树从岩石缝隙中长出来
    {"Sub": "bag", "Rel": "hanging from", "Obj": "branch", "Expected": "Yes"}    # 包挂在树枝上，视觉和物理上完全说得通
]

# 2. 匹配你原代码中的问题生成逻辑
def format_question(sub, rel, obj):
    sub_f = f'a group of {sub}' if sub in ['men', 'people'] else f'a {sub}'
    obj_f = f'a group of {obj}' if obj in ['men', 'people'] else f'a {obj}'
    
    rel_formatted = rel
    if rel != 'and':
        if rel == 'wears': rel_formatted = 'wearing'
        elif rel == 'says': rel_formatted = 'saying'
        elif rel == 'has': rel_formatted = 'have'
        
        rel_f = 'have' if rel_formatted == 'have' else f'be {rel_formatted}'
        return f"Can {sub_f} {rel_f} {obj_f}?"
    else:
        return f"Can {sub_f} {rel} {obj_f} appear together?"

# 3. 定义测试函数
def run_ablation(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 构建所有问题的 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 批量推理 (关闭 tqdm 防止 Jupyter 输出过长)
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    
    # 结果解析与统计
    results = []
    for case, out in zip(test_cases, outputs):
        answer = out.outputs[0].text.strip()
        # 简单判定：如果模型输出包含预期答案（忽略大小写）则判对
        is_correct = case['Expected'].lower() in answer.lower()
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Model_Output": answer,
            "Correct": is_correct
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"Finish!  {len(df_results['Correct'])} samples' accuracy: {accuracy:.1f}%\n")
    return df_results

# Experiment

In [4]:
temperature = 0.7

In [5]:
sampling_params = SamplingParams(
    temperature=temperature, 
    max_tokens=150, 
)

# Exp 1: Original prompts (v4)

In [6]:
prompt_v1 = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
Interpret this strictly as a statement about physical and commonsense plausibility in the real world.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
Respond with a Yes or No if both commonsense and visually plausible, without additional explainations."""

shots_v1 = [
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can an animal be eating an apple?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can a bird be under a mountain?"},
    {"role": "assistant", "content": "No"},
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "No"}
]

print("=== Exp 1 Original Prompt ===")
sampling_params = SamplingParams(
    temperature=temperature, 
    max_tokens=150, 
)
df_res1 = run_ablation(prompt_v1, shots_v1)
display(df_res1) 

=== Exp 1 Original Prompt ===
Finish!  50 samples' accuracy: 74.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,Yes,True
1,Can a person be wearing a hat?,Yes,Yes,True
2,Can a boat be flying in a room?,No,No,True
3,Can a cat be sitting on a table?,Yes,Yes,True
4,Can a pizza be eating a man?,No,No,True
5,Can a train be parked on a track?,Yes,Yes,True
6,Can a tree be riding a bike?,No,No,True
7,Can a dog be holding a branch?,Yes,Yes,True
8,Can a flower be growing on a rock?,Yes,Yes,True
9,Can a logo be of a banana?,Yes,Yes,True


# Exp 2: Chain-of-Thought (CoT)

In [7]:
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
End your response with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"},
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: 1) Cars are vehicles designed for roads. 2) This is a standard real-world activity. 3) It is visually plausible. \nFinal Answer: Yes"}
]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
sampling_params = SamplingParams(
    temperature=temperature, 
    max_tokens=150, 
)

print("=== Exp 2: Chain-of-Thought (CoT) ===")
df_res2 = run_ablation(prompt_cot, shots_cot)
display(df_res2)



=== Exp 2: Chain-of-Thought (CoT) ===
Finish!  50 samples' accuracy: 86.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,Reasoning: 1) Streets are designed for vehicles like cars. 2) This is a common real-world scenario. 3) It is visually plausible. \nFinal Answer: Yes,True
1,Can a person be wearing a hat?,Yes,Reasoning: 1) A hat is a common item of clothing. 2) People commonly wear them. 3) It is visually plausible. \nFinal Answer: Yes,True
2,Can a boat be flying in a room?,No,"Reasoning: 1) Boats are typically designed for water, not air. 2) They lack the necessary aerodynamics and propulsion for flight. 3) There is no feasible way for a boat to defy gravity in a room. 4) It would not be visually plausible in a real-world setting.\nFinal Answer: No",True
3,Can a cat be sitting on a table?,Yes,"Reasoning: 1) Cats are mobile animals. 2) They often climb onto various surfaces, including tables. 3) It's visually plausible to see a cat sitting on a table in a real-world setting. \nFinal Answer: Yes",True
4,Can a pizza be eating a man?,No,"Reasoning: 1) A pizza is a type of food, typically made of dough, sauce, cheese, and toppings. 2) In the real world, food items cannot consume or eat humans or other objects. 3) This situation is not physically or biologically possible. \nFinal Answer: No",True
5,Can a train be parked on a track?,Yes,Reasoning: 1) Trains are designed to operate on railway tracks. 2) Trains can be stopped and stopped trains are parked. 3) This is a common real-world scenario. 4) It is visually plausible.\nFinal Answer: Yes,True
6,Can a tree be riding a bike?,No,"Reasoning: 1) Trees are stationary plants, not mobile entities. 2) Riding a bike is a human activity that requires a living being with a physical structure. 3) It is physically and visually implausible for a tree to ride a bike. \nFinal Answer: No",True
7,Can a dog be holding a branch?,Yes,Reasoning: 1) Dogs have hands or paws that can grasp objects. 2) Branches are objects that can be grasped. 3) It is physically plausible and visually possible. \nFinal Answer: Yes,True
8,Can a flower be growing on a rock?,Yes,"Reasoning: 1) In the real world, plants like flowers require soil and adequate nutrients for growth. 2) Rocks do not provide these conditions. 3) However, it is possible for lichens, mosses, or other hardy plants to grow in small crevices or cracks on rocks in certain conditions. \nFinal Answer: Yes, but with a nuanced condition that the rock has a suitable environment, such as a microclimate or small crevices.",True
9,Can a logo be of a banana?,Yes,"Reasoning: 1) A logo is a symbol or design representing a company or brand. 2) A banana is a common object that can be used as a design element. 3) Logos can represent various objects, concepts, or brands. \nFinal Answer: Yes",True


In [8]:
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: An arm is a dependent biological part, not an independent object.\n- [Geometry]: Sky implies unattached aerial position. An arm lacks aerodynamic structure.\n- [Agency]: Arms cannot fly. This violates basic physics.\nFinal Answer: No"},
    
    {"role": "user", "content": "Can a person be wearing a hat?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A hat is sized to fit a person's head.\n- [Geometry]: 'Wearing' implies placed on the head. This is a standard spatial configuration.\n- [Agency]: A person has the physical capacity to wear items.\nFinal Answer: Yes"},
    
    {"role": "user", "content": "Can a vase be sitting on an airplane?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A vase is small and fragile. An airplane is massive.\n- [Geometry]: 'Sitting on' an airplane strictly means the exterior roof/wings. A vase cannot physically balance on the exterior of a plane under typical conditions.\n- [Agency]: Objects do not naturally sit on moving vehicle exteriors.\nFinal Answer: No"}
]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
sampling_params = SamplingParams(
    temperature=0., 
    max_tokens=150, 
)

print("=== Exp 3: Chain-of-Thought (CoT) v2 ===")
df_res2 = run_ablation(prompt_cot, shots_cot)
display(df_res2)




=== Exp 3: Chain-of-Thought (CoT) v2 ===
Finish!  50 samples' accuracy: 90.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,"Reasoning:\n- [Scale/Part]: A car is designed for street use.\n- [Geometry]: A car is designed to be placed on a road, which is a common street feature.\n- [Agency]: Cars are designed to be driven on streets.\nFinal Answer: Yes",True
1,Can a person be wearing a hat?,Yes,This is the same question as the one you asked earlier. The answer remains the same.\n\n Reasoning:\n- [Scale/Part]: A hat is sized to fit a person's head.\n- [Geometry]: 'Wearing' implies placed on the head. This is a standard spatial configuration.\n- [Agency]: A person has the physical capacity to wear items.\nFinal Answer: Yes,True
2,Can a boat be flying in a room?,No,"Reasoning:\n- [Scale/Part]: A boat is a large object, not designed for indoor spaces.\n- [Geometry]: A room is a confined space, not a body of water. A boat requires water to float.\n- [Agency]: Boats do not have the capability to defy gravity and float in air.\nFinal Answer: No",True
3,Can a cat be sitting on a table?,Yes,"Reasoning:\n- [Scale/Part]: A cat is small enough to fit on a table.\n- [Geometry]: 'Sitting on' implies a stable, flat surface. Tables are designed for this purpose.\n- [Agency]: Cats have the physical capability to sit.\nFinal Answer: Yes",True
4,Can a pizza be eating a man?,No,"Reasoning:\n- [Scale/Part]: A pizza is a food item, a man is a living being. The size difference is significant.\n- [Geometry]: 'Eating' implies the pizza is consuming the man, which is biologically impossible.\n- [Agency]: Pizzas do not have the capacity to consume living beings.\nFinal Answer: No",True
5,Can a train be parked on a track?,Yes,"Reasoning:\n- [Scale/Part]: A train is a large vehicle, and a track is designed for it.\n- [Geometry]: A train is designed to be placed on a track, and the track is designed to support it.\n- [Agency]: Trains are capable of being parked on tracks.\nFinal Answer: Yes",True
6,Can a tree be riding a bike?,No,Reasoning:\n- [Scale/Part]: A tree is much larger and heavier than a bike.\n- [Geometry]: A tree's size and structure do not allow for sitting on a bike.\n- [Agency]: Trees do not have the physical capability to operate a bike.\nFinal Answer: No,True
7,Can a dog be holding a branch?,Yes,Reasoning:\n- [Scale/Part]: A dog is capable of holding objects in its mouth.\n- [Geometry]: A branch can fit in a dog's mouth.\n- [Agency]: Dogs have the physical capability to grasp and hold objects.\nFinal Answer: Yes,True
8,Can a flower be growing on a rock?,Yes,"Reasoning:\n- [Scale/Part]: A flower is relatively small, and a rock can be large enough to support it.\n- [Geometry]: 'Growing on' implies attachment to the rock's surface. This is a common spatial configuration.\n- [Agency]: Plants have the capability to grow in various environments, including rocks.\nFinal Answer: Yes",True
9,Can a logo be of a banana?,Yes,"Reasoning:\n- [Scale/Part]: A logo can be any size, but a banana is a small object.\n- [Geometry]: A logo can be any shape, and a banana is a specific shape. This is geometrically possible.\n- [Agency]: A logo is a design, which can be created to depict any object, including a banana.\nFinal Answer: Yes",True


In [9]:
from collections import Counter
import re
import pandas as pd

# 1. 重新配置 Self-consistency 的采样参数
# 【关键改动】
# temperature: 调高到 0.7 引入随机性，确保模型每次的思维路径不一样
# n=5: 让 vLLM 一次性并发生成 5 个不同的回答，极大地节省时间
sampling_params_sc = SamplingParams(temperature=0.1, top_p=0.95, n=5, max_tokens=256)

# 复用你设计得非常好的 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = []

# 2. 定义专属的 Self-Consistency 运行函数
def run_self_consistency(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 组装所有 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 并行生成（此时 outputs 中的每个 out 将包含 5 个生成结果）
    outputs = llm.generate(prompts, sampling_params_sc, use_tqdm=False)
    
    results = []
    for case, out in zip(test_cases, outputs):
        votes = []
        
        # 遍历同一个 prompt 生成的 5 个不同推理过程
        for o in out.outputs:
            text = o.text.strip()
            
            # 使用正则严格匹配你设定的 Final Answer: Yes/No 格式
            match = re.search(r"Final Answer:\s*(Yes|No)", text, re.IGNORECASE)
            if match:
                votes.append(match.group(1).capitalize()) # 统一转为 Yes 或 No
            else:
                # 兜底逻辑：如果模型没按格式输出，从最后 20 个字符找 yes/no
                last_chars = text.lower()[-20:]
                if 'yes' in last_chars: votes.append('Yes')
                elif 'no' in last_chars: votes.append('No')

        # 多数投票机制 (Majority Voting)
        if not votes:
            final_vote = "Error"
        else:
            vote_counts = Counter(votes)
            # 取出得票数最高的答案 (例如: [('No', 4), ('Yes', 1)])
            final_vote = vote_counts.most_common(1)[0][0] 
            
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Votes_Detail": str(dict(Counter(votes))), # 方便你观察票数分布 (例如 "{'No': 4, 'Yes': 1}")
            "Final_Vote": final_vote,
            "Correct": final_vote.lower() == case['Expected'].lower()
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\n=> Self-Consistency 测试完成! 准确率为: {accuracy:.1f}%\n")
    return df_results

# 3. 运行测试
print("=== Exp 3: Self-Consistency (SC) ===")
pd.set_option('display.max_colwidth', None)
df_res_sc = run_self_consistency(prompt_cot, shots_cot)
display(df_res_sc)

=== Exp 3: Self-Consistency (SC) ===

=> Self-Consistency 测试完成! 准确率为: 70.0%



,Question,Expected,Votes_Detail,Final_Vote,Correct
0,Can a car be on a street?,Yes,{'Yes': 5},Yes,True
1,Can a person be wearing a hat?,Yes,{'Yes': 5},Yes,True
2,Can a boat be flying in a room?,No,{'No': 5},No,True
3,Can a cat be sitting on a table?,Yes,{'Yes': 5},Yes,True
4,Can a pizza be eating a man?,No,{'No': 5},No,True
5,Can a train be parked on a track?,Yes,{'Yes': 5},Yes,True
6,Can a tree be riding a bike?,No,{'No': 5},No,True
7,Can a dog be holding a branch?,Yes,{'Yes': 5},Yes,True
8,Can a flower be growing on a rock?,Yes,{'Yes': 5},Yes,True
9,Can a logo be of a banana?,Yes,{'Yes': 5},Yes,True


In [10]:
from collections import Counter
import re
import pandas as pd

# 1. 重新配置 Self-consistency 的采样参数
# 【关键改动】
# temperature: 调高到 0.7 引入随机性，确保模型每次的思维路径不一样
# n=5: 让 vLLM 一次性并发生成 5 个不同的回答，极大地节省时间
sampling_params_sc = SamplingParams(temperature=0.1, top_p=0.95, n=5, max_tokens=256)

# 复用你设计得非常好的 CoT Prompt
prompt_cot = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world and could appear in a real photo.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
[OUTPUT FORMAT]:
Reasoning:
- [Scale/Part]: <Size or hierarchy check>
- [Geometry]: <Physical alignment/spatial check>
- [Agency]: <Capability check>
Final Answer: Yes/No"""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: An arm is a dependent biological part, not an independent object.\n- [Geometry]: Sky implies unattached aerial position. An arm lacks aerodynamic structure.\n- [Agency]: Arms cannot fly. This violates basic physics.\nFinal Answer: No"},
    
    {"role": "user", "content": "Can a person be wearing a hat?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A hat is sized to fit a person's head.\n- [Geometry]: 'Wearing' implies placed on the head. This is a standard spatial configuration.\n- [Agency]: A person has the physical capacity to wear items.\nFinal Answer: Yes"},
    
    {"role": "user", "content": "Can a vase be sitting on an airplane?"},
    {"role": "assistant", "content": "Reasoning:\n- [Scale/Part]: A vase is small and fragile. An airplane is massive.\n- [Geometry]: 'Sitting on' an airplane strictly means the exterior roof/wings. A vase cannot physically balance on the exterior of a plane under typical conditions.\n- [Agency]: Objects do not naturally sit on moving vehicle exteriors.\nFinal Answer: No"}
]

# 2. 定义专属的 Self-Consistency 运行函数
def run_self_consistency(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 组装所有 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 并行生成（此时 outputs 中的每个 out 将包含 5 个生成结果）
    outputs = llm.generate(prompts, sampling_params_sc, use_tqdm=False)
    
    results = []
    for case, out in zip(test_cases, outputs):
        votes = []
        
        # 遍历同一个 prompt 生成的 5 个不同推理过程
        for o in out.outputs:
            text = o.text.strip()
            
            # 使用正则严格匹配你设定的 Final Answer: Yes/No 格式
            match = re.search(r"Final Answer:\s*(Yes|No)", text, re.IGNORECASE)
            if match:
                votes.append(match.group(1).capitalize()) # 统一转为 Yes 或 No
            else:
                # 兜底逻辑：如果模型没按格式输出，从最后 20 个字符找 yes/no
                last_chars = text.lower()[-20:]
                if 'yes' in last_chars: votes.append('Yes')
                elif 'no' in last_chars: votes.append('No')

        # 多数投票机制 (Majority Voting)
        if not votes:
            final_vote = "Error"
        else:
            vote_counts = Counter(votes)
            # 取出得票数最高的答案 (例如: [('No', 4), ('Yes', 1)])
            final_vote = vote_counts.most_common(1)[0][0] 
            
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Votes_Detail": str(dict(Counter(votes))), # 方便你观察票数分布 (例如 "{'No': 4, 'Yes': 1}")
            "Final_Vote": final_vote,
            "Correct": final_vote.lower() == case['Expected'].lower()
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\n=> Self-Consistency 测试完成! 准确率为: {accuracy:.1f}%\n")
    return df_results

# 3. 运行测试
print("=== Exp 3: Self-Consistency (SC) ===")
pd.set_option('display.max_colwidth', None)
df_res_sc = run_self_consistency(prompt_cot, shots_cot)
display(df_res_sc)

=== Exp 3: Self-Consistency (SC) ===

=> Self-Consistency 测试完成! 准确率为: 84.0%



,Question,Expected,Votes_Detail,Final_Vote,Correct
0,Can a car be on a street?,Yes,{'Yes': 5},Yes,True
1,Can a person be wearing a hat?,Yes,{'Yes': 5},Yes,True
2,Can a boat be flying in a room?,No,{'No': 5},No,True
3,Can a cat be sitting on a table?,Yes,{'Yes': 5},Yes,True
4,Can a pizza be eating a man?,No,{'No': 5},No,True
5,Can a train be parked on a track?,Yes,{'Yes': 5},Yes,True
6,Can a tree be riding a bike?,No,{'No': 5},No,True
7,Can a dog be holding a branch?,Yes,{'Yes': 5},Yes,True
8,Can a flower be growing on a rock?,Yes,{'Yes': 5},Yes,True
9,Can a logo be of a banana?,Yes,{'Yes': 5},Yes,True


ERROR 05-14 16:51:08 [core_client.py:667] Engine core proc EngineCore died unexpectedly, shutting down client.


/home/shsi/.conda/envs/promptllm/lib/python3.10/multiprocessing/resource_tracker.py:224: UserWarning: resource_tracker: There appear to be 1 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
